<a target="_blank" href="https://colab.research.google.com/github/wilhelm-lab/oktoberfest/blob/notebook_workshop/tutorials/EuBIC_2026_Winterschool_Oktoberfest_Workshop.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Rescoring with Oktoberfest Workshop

This notebook is prepared to be run in Google [Colaboratory](https://colab.research.google.com/).

This notebook contains tasks that are designed to guide new users through the following topics:

1. How to install oktoberfest and load packages
2. How to get the required data
3. How to prepare a configuration file & run a job 
4. How to interpret the output

if time allows:

5. Re-creating protein groups with Picked Group FDR

for you to check out later:

6. explore another Oktoberfest functionality: Spectral Library Generation
7. explore another Oktoberfest functionality: Collision Energy Calibration

# 1. Installation

Before using Oktoberfest, the package and dependencies need to be installed. This step is only required once on your notebook, but it may need to be repeated upon reloading Google Colab.

## Task 1.1

What are the requirements for Oktoberfest and where do you find this information? (Hint: Search the Oktoberfest documentation at readthedocs).

## Task 1.2

Before you execute anything, set the google colab runtime version to 2026.01. This makes sure a matching python version is available.

Execute the below code cells, which installs percolator and Oktoberfest and restart the session if asked.

In [ ]:
!python --version

In [ ]:
!wget https://github.com/percolator/percolator/releases/download/rel-3-06-01/percolator-v3-06-linux-amd64.deb
!dpkg -i percolator-v3-06-linux-amd64.deb
!pip install -q "numpy==2.1.3"
!pip install -q oktoberfest
!pip install -q networkx grpcio

## Task 1.3

For this notebook to work, a few packages need to be imported that provide the functions used in the following. They should already be installed as dependencies of Oktoberfest. Should you get an error here, check that installation of the required packages was successful.

Import the below packages and functions by executing the code in the cell.

In [ ]:
from oktoberfest.runner import run_job
from oktoberfest import __version__ as version
import os
import json
import urllib.request
import requests
import shutil
from tqdm.auto import tqdm
import pandas as pd

If this works, you have installed Oktoberfest correctly.


## Task 1.4
How can you check that you are using the current stable version?

In [ ]:
version

# 2: Getting the example data

The data used in this notebook is provided as a zip archive that can be downloaded from zenodo. You can check it out here: https://zenodo.org/records/10814834

## Task 2.1

Download and unpack the data using the code cells below. You should see a progress bar while it is downloading the file (86MB, approx. 1 minute).

In [ ]:
url = "https://zenodo.org/records/10814834/files/tomato_dataset_example.zip"  # download link from public zenodo record
download_dir = "Oktoberfest_input/"                                           # local download directory
file_name = "tomato_dataset_example.zip"                                      # local file name for the downloaded file

In [ ]:
if not os.path.isdir(download_dir):
  os.mkdir(download_dir)
download_file = os.path.join(download_dir, file_name)
filesize = int(requests.head(url).headers.get('content-length', -1))
with tqdm(unit="B", total=filesize, unit_scale=True, unit_divisor=1000, miniters=1, desc=url.split("/")[-1]) as t:
    urllib.request.urlretrieve(url=url, filename=download_file, reporthook=lambda blocks, block_size, _: t.update(blocks * block_size - t.n))
shutil.unpack_archive(download_file, download_dir)

## Task 2.2

Check that the download was successful. Hint: Use the file browser on the left side to search for the folder you defined using the __download_dir__ variable above and check the content.  

What do you find here?

# 3: Rescoring with Oktoberfest

The main feature of oktoberfest is to perform rescoring. This requires two main inputs:
- unfiltered search results, for MaxQuant, this would mean a run with 100% PSM and peptide FDR
- get spectra, either in ThermoFisher .RAW, Bruker .d, or mzML format

In addition, Oktoberfest can get predictions from various data dependent models, that are provided by a Koina instance.

## Task 3.1

Where do you find information about the configuration options, example configurations, and the supported prediction models? (Hint: Check the [Usage principles](https://oktoberfest.readthedocs.io/en/latest/usage.html) in the Oktoberfest documentation) 

The data we are working with here was aquired using beam-type collision induced dissociation (HCD) without tandem mass tags (TMT). Which are the models to use for fragment intensity prediction and retention time prediction and what is the server URL that provides access to these models?

Also specify the directory you want to store all the outputs from Oktoberfest in.

Define below variables accordingly.

In [ ]:
spectra = "Oktoberfest_input/tomato_dataset_example/5407_GC4_063119_S00_U4_R1.mzML"  # this is the location of the mzML file containing the measured spectra
spectra_type =  "mzml"                                                               # this is the format the spectra are provided in ("mzml", "RAW", "d")

search_results = "Oktoberfest_input/tomato_dataset_example/msms.txt"                 # this is the location of the search engine output
search_results_type = "maxquant"                                                     # this is the name of the search engine that produced the search results

intensity_model =  "Prosit_2020_intensity_HCD"      # this is the model used for fragment intensity prediction
retention_time_model = "Prosit_2019_irt"            # this is the model used for retention time prediction
prediction_server =  "koina.wilhelmlab.org:443"     # the Koina server that provides access to the specified models

output_directory = "rescore_out"                    # this is the output folder for everything Oktoberfest produces during rescoring

## Task 3.2

Save the variables you have defined above in the configuration dictionary below and store it to disk. For simplicity, this is providing a minimal configuration for this task, so you can simply execute the code cell.

A detailed explanation of all available configuration options can be found in the [Usage principles](https://oktoberfest.readthedocs.io/en/latest/usage.html) in the Oktoberfest documentation.

The oktoberfest documentation provides [example configurations](https://oktoberfest.readthedocs.io/en/latest/jobs.html#c-rescoring) that show you how a typical rescoring run for MaxQuant is set up with all the available options.

If you want to get detailed information about individual options and allowed values, you can check the documentation for the [full configuration](https://oktoberfest.readthedocs.io/en/latest/config.html).

In [ ]:
task_config_rescoring = {
    "type": "Rescoring",
    "inputs":{
        "search_results": search_results,
        "search_results_type": search_results_type,
        "spectra": spectra,
        "spectra_type": spectra_type
    },
    "output": output_directory,
    "models": {
        "intensity": intensity_model,
        "irt": retention_time_model
    },
    "prediction_server": prediction_server,
    "ssl": True,
    "numThreads": 1,
    "fdr_estimation_method": "percolator",
    "massTolerance": 20,
    "unitMassTolerance": "ppm"
}

# this is for storing the file on disk
with open('./rescoring_config.json', 'w') as fp:
    json.dump(task_config_rescoring, fp)

## Task 3.3

Start the rescoring run.

After preparation of the configuration file, oktoberfest can be instructed to run a job with the provided configuration file. This step takes a while (approx. 3-5 minutes) and provides you with log output that tracks the progress of rescoring.

Oktoberfest will perform the following steps:

- read the search results from maxquant and translate them to the internal format used by Oktoberfest. The specification for this format can be found in the documentation.
- parse the mzml data to retreive MS2 spectra, then merge with the search results to generate PSMs, filtering out spectra without a search result
- annotation of spectra for all y- and b-fragments in charge states 1-3
- perform a normalized collision energy (NCE) calibration using the top 1000 highest scoring target PSMs, to determine the NCE for which the highest spectral angle can be achieved
- fragment intensity and retention time prediction for all PSMs
- retention time alignment, spectral angle and further feature calculation for rescoring using percolator
- rescoring using features from intensity and retention time prediction and the original search engine score
- plotting summaries of the rescoring run

In [ ]:
run_job("./rescoring_config.json")

# 4. The rescoring results

Explore the output folder of Oktoberfest using the file browser on the left.

Where do you find information about the output folder structure and what you can find where? (Hint: Check the [Generated Outputs](https://oktoberfest.readthedocs.io/en/stable/outputs.html) page in the Oktoberfest documentation)

## Task 4.1

Based on the created plots, answer the following questions:
* What is the optimal collision energy?
* How many more peptides are confidently identified after rescoring?
* How does rescoring affect the score distributions for targets & decoys on peptide level? Compare original and rescore.

----------------------------------------

# if time allows:

# 5: Re-creating protein groups

After rescoring the data, we need to re-assemble the peptides to protein groups. To do so, we can use [Picked Group FDR](https://github.com/kusterlab/picked_group_fdr) python package that is one of the most sophisticated solutions for this purpose.

For more informtion and usage principle, please check out Picked Group FDR's [readthedocs](https://picked-group-fdr.readthedocs.io/en/latest/) page.

## Task 5.1

Instal the Picked Group FDR package from the deveopment branch of its GitHub repository.

In [ ]:
!git clone https://github.com/kusterlab/picked_group_fdr.git
!pip install -q ./picked_group_fdr

## Task 5.2

Import the required modules.

In [ ]:
import picked_group_fdr.pipeline as picked_group_fdr
from picked_group_fdr.digestion_params import DigestionParams

## Task 5.3

Update the MaxQuant evidence.txt file with the Oktoberfest output.

In [ ]:
picked_group_fdr.run_update_evidence(
    ["Oktoberfest_input/tomato_dataset_example/evidence.txt"],   # the original evidence.txt file from MaxQuant
    ["rescore_out/results/percolator/rescore.percolator.psms.txt", "rescore_out/results/percolator/rescore.percolator.decoy.psms.txt"],  # the Oktoberfest rescoring results
    ["rescore_out/results/evidence.txt"],   # the updated evidence file
    "prosit",
    suppress_missing_peptide_warning=True
)

## Task 5.4

Run the Picked Group FDR algorithm

After this step you will have the new protein groups file. Be mindful that the output is not filtered for 1% FDR.

In [ ]:
picked_group_fdr.run_picked_group_fdr(
    ["rescore_out/results/evidence.txt"],  # the updated evidence file
    "rescore_out/results/proteinGroups_Unfiltered.txt",  # the output file for the generated protein groups (no FDR filtering)
    ["Oktoberfest_input/tomato_dataset_example/UP000004994.fasta"],  # the original FASTA file used for the search
    [DigestionParams("trypsinp", "full", 7, 30, 2, "KR", False)],  # the digestion parameters used for the search
    True,   # do quantification
    1,  # LFQ minimum peptide ratios
    suppress_missing_peptide_warning=True,
)

## Task 5.5

Filtering the protein groups with 1% FDR.

In [ ]:
picked_group_fdr.run_filter_fdr_maxquant(
    ["rescore_out/results/proteinGroups_Unfiltered.txt"],  # the protein groups file generated in the previous step (no FDR filtering)
    "rescore_out/results/proteinGroups_1%FDR.txt",  # the output file for the FDR-filtered protein groups (1% FDR)
    fdr_cutoff=0.01  # the FDR cutoff for filtering the protein groups
)

## Task 5.6

Explore the output of Picked Group FDR using the file browser on the left. Load the FDR-filtered table and explore it.

In [ ]:
protein_groups = pd.read_csv("rescore_out/results/proteinGroups_1%FDR.txt", sep="\t")
print(f"Protein groups after FDR filtering: {len(protein_groups)}")
protein_groups.head(10)

--------------

# 6: explore another Oktoberfest functionality: Spectral Library Generation

More info on the required parameters can be found [here](https://oktoberfest.readthedocs.io/en/stable/config.html#applicable-to-spectral-library-generation).

*this task takes quite long*

## Task 6.1

Generate an Oktoberfest config dictionary and save it as a file to disk

In [ ]:
task_config_spectral_lib = {
    "type": "SpectralLibraryGeneration",
    "tag": "",
    "inputs": {
        "library_input": "Oktoberfest_input/tomato_dataset_example/UP000004994.fasta", # the original FASTA file
        "library_input_type": "fasta",
        "instrument_type": ""
    },
    "output": "./spectral_library_out",
    "models": {
        "intensity": "Prosit_2020_intensity_HCD",
        "irt": "Prosit_2019_irt"
    },
    "prediction_server": "koina.wilhelmlab.org:443",
    "ssl": True,
    "numThreads": 1,
    "spectralLibraryOptions": {
        "fragmentation": "HCD",
        "collisionEnergy": 30,
        "precursorCharge": [2,3],
        "minIntensity": 5e-4,
        "batchsize": 10000,
        "format": "msp",
        "nrOx": 1,
    },
    "fastaDigestOptions": {
        "digestion": "full",
        "missedCleavages": 2,
        "minLength": 7,
        "maxLength": 30,
        "enzyme": "trypsin",
        "specialAas": "KR",
        "db": "concat"
    },
}

# this is for storing the file on disk
with open('./spectral_library_config.json', 'w') as fp:
    json.dump(task_config_spectral_lib, fp)

## Task 6.2

Run the spectral library generation Oktoberfest job

In [ ]:
run_job("./spectral_library_config.json")

## Task 6.3

Explore the created results using the file browser on the left. 

--------------

# 7: explore another Oktoberfest functionality: Collision Energy Calibration

The CE calibration will be performed on the top 1000 PSMs (based on the andromeda score in the msms.txt).

## Task 7.1

Generate an Oktoberfest config dictionary and save it as a file to disk

In [ ]:
task_config_ce_calibration = {
    "type": "CollisionEnergyCalibration",
    "tag": "",
    "inputs":{
        "search_results": "Oktoberfest_input/tomato_dataset_example/msms.txt",
        "search_results_type": "Maxquant",
        "spectra": "Oktoberfest_input/tomato_dataset_example/5407_GC4_063119_S00_U4_R1.mzML",
        "spectra_type": "mzml"
    },
    "output": "./ce_calibration_out",
    "models": {
        "intensity": "Prosit_2020_intensity_HCD",
        "irt": "Prosit_2019_irt"
    },
    "prediction_server": "koina.wilhelmlab.org:443",
    "ssl": True,
    "massTolerance": 20,
    "unitMassTolerance": "ppm",
    "numThreads": 1
}

# this is for storing the file on disk
with open('./ce_calibration_config.json', 'w') as fp:
    json.dump(task_config_ce_calibration, fp)

## Task 7.2

Run the CE Calibration Oktoberfest job.

In [ ]:
run_job("./ce_calibration_config.json")

## Task 7.3

Explore the created results using the file browser on the left. 